In [ ]:
# EXPERIMENT 10: L2, DROPOUT AND EARLY STOPPING COMPARISON

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

np.random.seed(42)
tf.random.set_seed(42)

# 1. CREATE A SAMPLE DIABETES DATASET
X, y = make_classification(
    n_samples=500,
    n_features=8,
    n_informative=5,
    n_redundant=1,
    n_classes=2,
    weights=[0.55, 0.45],
    random_state=42
)

feature_names = [
    'Glucose', 'BMI', 'Age', 'Insulin',
    'BloodPressure', 'Pregnancies', 'SkinThickness', 'DiabetesPedigree'
]

df = pd.DataFrame(X, columns=feature_names)
df['Diabetes'] = y

print("Sample Dataset:")
print(df.head())

# 2. PREPROCESS THE DATA
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# 3. BUILD MLP MODELS
def build_model(l2_value=None, dropout_rate=None):

    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(8,)),
        tf.keras.layers.Dense(
            128, activation='relu',
            kernel_regularizer=tf.keras.regularizers.l2(l2_value)
            if l2_value else None
        ),
        tf.keras.layers.Dense(
            64, activation='relu',
            kernel_regularizer=tf.keras.regularizers.l2(l2_value)
            if l2_value else None
        ),

        tf.keras.layers.Dropout(dropout_rate)
        if dropout_rate else tf.keras.layers.Activation('linear'),

        tf.keras.layers.Dense(32, activation='relu'),
        tf.keras.layers.Dense(1, activation='sigmoid')
    ])

    model.compile(
        optimizer='adam',
        loss='binary_crossentropy',
        metrics=['accuracy']
    )

    return model

results = []
histories = {}

# BASELINE MODEL
print("\nTraining Baseline Model...")
baseline = build_model()

h = baseline.fit(
    X_train, y_train,
    epochs=100,
    batch_size=16,
    validation_split=0.2,
    verbose=0
)

histories['Baseline'] = h

train_acc = baseline.evaluate(X_train, y_train, verbose=0)[1]
test_acc = baseline.evaluate(X_test, y_test, verbose=0)[1]

results.append(['Baseline', train_acc, test_acc, train_acc-test_acc])

# 4. L2 REGULARIZATION
l2_values = [0.001, 0.01, 0.1]

for value in l2_values:
    print(f"\nTraining L2 Model: {value}")

    model = build_model(l2_value=value)

    h = model.fit(
        X_train, y_train,
        epochs=100,
        batch_size=16,
        validation_split=0.2,
        verbose=0
    )

    histories[f'L2 {value}'] = h

    train_acc = model.evaluate(X_train, y_train, verbose=0)[1]
    test_acc = model.evaluate(X_test, y_test, verbose=0)[1]

    results.append([f'L2 {value}', train_acc, test_acc, train_acc-test_acc])

# 5. DROPOUT
dropout_values = [0.2, 0.3, 0.5]

for rate in dropout_values:
    print(f"\nTraining Dropout Model: {rate}")

    model = build_model(dropout_rate=rate)

    h = model.fit(
        X_train, y_train,
        epochs=100,
        batch_size=16,
        validation_split=0.2,
        verbose=0
    )

    histories[f'Dropout {rate}'] = h

    train_acc = model.evaluate(X_train, y_train, verbose=0)[1]
    test_acc = model.evaluate(X_test, y_test, verbose=0)[1]

    results.append([f'Dropout {rate}', train_acc, test_acc, train_acc-test_acc])

# 6. EARLY STOPPING
print("\nTraining Early Stopping Model...")

early_model = build_model()

early_callback = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True
)

h = early_model.fit(
    X_train, y_train,
    epochs=100,
    batch_size=16,
    validation_split=0.2,
    callbacks=[early_callback],
    verbose=0
)

histories['Early Stopping'] = h

train_acc = early_model.evaluate(X_train, y_train, verbose=0)[1]
test_acc = early_model.evaluate(X_test, y_test, verbose=0)[1]

results.append(['Early Stopping', train_acc, test_acc, train_acc-test_acc])

# 7. PLOT BASELINE OVERFITTING
plt.figure(figsize=(8, 5))
plt.plot(histories['Baseline'].history['accuracy'], label='Training Accuracy')
plt.plot(histories['Baseline'].history['val_accuracy'], label='Validation Accuracy')
plt.title('Baseline MLP - Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(8, 5))
plt.plot(histories['Baseline'].history['loss'], label='Training Loss')
plt.plot(histories['Baseline'].history['val_loss'], label='Validation Loss')
plt.title('Baseline MLP - Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)
plt.show()

# 8. PLOT L2 COMPARISON
plt.figure(figsize=(8, 5))

for name in ['L2 0.001', 'L2 0.01', 'L2 0.1']:
    plt.plot(histories[name].history['val_loss'], label=name)

plt.title('L2 Regularization - Validation Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)
plt.show()

# 9. PLOT DROPOUT COMPARISON
plt.figure(figsize=(8, 5))

for name in ['Dropout 0.2', 'Dropout 0.3', 'Dropout 0.5']:
    plt.plot(histories[name].history['val_loss'], label=name)

plt.title('Dropout - Validation Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)
plt.show()

# 10. PLOT EARLY STOPPING
plt.figure(figsize=(8, 5))
plt.plot(histories['Early Stopping'].history['loss'], label='Training Loss')
plt.plot(histories['Early Stopping'].history['val_loss'], label='Validation Loss')
plt.title('Early Stopping - Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)
plt.show()

# 11. RESULTS TABLE
results_df = pd.DataFrame(
    results,
    columns=['Model', 'Train Accuracy', 'Test Accuracy', 'Train-Test Gap']
)

print("\nMODEL COMPARISON:")
print(results_df.round(4).to_string(index=False))

# 12. COMPARISON GRAPH
plt.figure(figsize=(10, 5))
plt.bar(results_df['Model'], results_df['Test Accuracy'])
plt.title('Test Accuracy Comparison')
plt.xlabel('Models')
plt.ylabel('Test Accuracy')
plt.xticks(rotation=45, ha='right')
plt.grid(axis='y')
plt.tight_layout()
plt.show()

print("\nExperiment 10 Completed Successfully!")

Sample Dataset:
    Glucose       BMI       Age   Insulin  BloodPressure  Pregnancies  \
0  0.682069 -0.466528  1.702391  2.364848      -1.237862    -2.329252   
1 -1.130615 -0.628521  1.897531  0.914287       0.058928    -0.519921   
2 -1.466785 -0.776843 -1.588155  0.785985      -2.824823     1.112359   
3  0.458168 -2.569133  0.587566 -1.369820      -0.339987    -0.340855   
4  1.476934 -0.699262  0.752082  1.060880       1.123465     3.998698   

   SkinThickness  DiabetesPedigree  Diabetes  
0       0.126178          3.646918         0  
1       0.622711          1.496569         0  
2      -0.501402         -1.293043         0  
3      -0.979721         -1.384609         0  
4      -1.167780         -1.697894         0  

Training Baseline Model...

Training L2 Model: 0.001

Training L2 Model: 0.01

Training L2 Model: 0.1

Training Dropout Model: 0.2

Training Dropout Model: 0.3

Training Dropout Model: 0.5

Training Early Stopping Model...
